# BioNodulo on Google Colab

This notebook launches a temporary BioNodulo instance inside a Colab runtime. Files created in Colab are ephemeral unless you download or save them elsewhere.

BioNodulo is paid software distributed under the BioNodulo Closed Alpha Commercial License. Closed-alpha access is limited to authorized users and institutions, and BioNodulo may not be freely redistributed.

In [ ]:
%cd /content
!apt-get update -qq && apt-get install -y -qq bubblewrap uidmap
!command -v bwrap && command -v newuidmap && command -v newgidmap
!test -d BioNodulo || git clone -q --branch main https://github.com/Classacre/BioNodulo.git
%cd /content/BioNodulo
!git fetch -q origin main
!git checkout -q main
!git reset --hard -q origin/main
!echo 'BioNodulo git version:'
!git status -sb
!git log -1 --oneline --decorate
!python -m pip install -q .
!test -x /root/.pixi/bin/pixi || PIXI_NO_PATH_UPDATE=1 PIXI_HOME=/root/.pixi PIXI_BIN_DIR=/root/.pixi/bin bash -c "curl -fsSL https://pixi.sh/install.sh | bash"
import re
import shutil
import subprocess
from pathlib import Path

def run(cmd, *, cwd=None):
    print('+', ' '.join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)

def node_version_ok():
    node = shutil.which('node')
    if node is None:
        return False
    version = subprocess.check_output([node, '--version'], text=True).strip()
    print('node', version, flush=True)
    match = re.match(r'v(\d+)\.(\d+)\.', version)
    if not match:
        return False
    major, minor = map(int, match.groups())
    return major > 20 or (major == 20 and minor >= 19)

if not node_version_ok():
    run(['bash', '-lc', 'curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs'])

run(['node', '--version'])
run(['npm', '--version'])
# npm install (not ci): ci requires a perfect lockfile match and fails on
# platform-specific optional dep resolution differences between the machine
# that generated the lockfile and the Colab runtime.
run(['npm', 'install'], cwd='web')
run(['npm', 'run', 'build'], cwd='web')
assert Path('web/dist/assets').is_dir(), 'Frontend build incomplete: web/dist/assets missing. Check the npm build output above.'

## Start BioNodulo through Cloudflare Tunnel

Run this cell and wait for a `trycloudflare.com` URL to print. Open that URL to use BioNodulo while this cell keeps running. BioNodulo starts offline; use the Collaboration menu in the app to create or join a temporary share link.

The tunnel URL is public to anyone who has the link. Do not use it for sensitive data. Stop the cell when you are done to stop the BioNodulo server.

In [ ]:
!wget -q -O /content/cloudflared-linux-amd64.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared-linux-amd64.deb >/dev/null

import socket
import threading
import time
from pathlib import Path

assert Path('web/dist/assets').is_dir(), 'Missing web/dist/assets. Re-run the setup/build cell before starting BioNodulo.'

workspace = Path('/content/bionodulo_workspace')
workspace.mkdir(exist_ok=True)
def launch_cloudflare_tunnel(port):
    while True:
        time.sleep(0.5)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(('127.0.0.1', port)) == 0:
                break

    print('\nBioNodulo is ready. Launching Cloudflare Tunnel...\n')
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    for raw_line in tunnel.stderr:
        line = raw_line.decode(errors='replace')
        if 'trycloudflare.com' in line:
            url = line[line.find('http'):].strip()
            print(f'Open BioNodulo: {url}')
            print('Create a collaboration link from the top-right Collaboration menu when you want to invite others.')
            print('Keep this cell running while you use the app.')

threading.Thread(target=launch_cloudflare_tunnel, daemon=True, args=(8000,)).start()

!python main.py --host 0.0.0.0 --port 8000 --project-root /content/bionodulo_workspace